# Titanic Survival Prediction

**Objective**: Predict whether a passenger survived the Titanic disaster using PyTorch neural networks.

| Task | Description |
|------|-------------|
| 1 | Linear baseline model (Logistic Regression via `nn.Linear`) |
| 2 | Non-linear MLP with hidden layers and ReLU activations |
| 3 | Compare: Accuracy, Precision, Recall, F1-score |
| 4 | Feature engineering: FamilySize, Title extraction |

In [ ]:

# pandas/numpy   : data loading and manipulation
# torch / nn     : tensors, training, layers and loss functions
# sklearn        : train/test split and evaluation metrics
# matplotlib/seaborn : visualisation
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

torch.manual_seed(42)
np.random.seed(42)


---
## Step 1 — Load and Explore the Data

In [ ]:

## Load the data into a dataframe
df = pd.read_csv('./train.csv')
print(f'Shape: {df.shape}')
df.head()


In [ ]:

# Check missing values and survival distribution
print('Missing values per column:')
print(df.isnull().sum())
print('Survival counts:')
print(df['Survived'].value_counts())
print(f'Survival rate: {df["Survived"].mean():.2%}')


In [ ]:

# Visualise survival rate by Pclass, Sex and Age distribution
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

df.groupby('Pclass')['Survived'].mean().plot(kind='bar', ax=axes[0],
    color=['#e74c3c','#f39c12','#2ecc71'], edgecolor='black')
axes[0].set_title('Survival Rate by Pclass')
axes[0].set_ylabel('Survival Rate')
axes[0].set_xticklabels(['1st','2nd','3rd'], rotation=0)

df.groupby('Sex')['Survived'].mean().plot(kind='bar', ax=axes[1],
    color=['#3498db','#e91e8c'], edgecolor='black')
axes[1].set_title('Survival Rate by Sex')
axes[1].set_ylabel('Survival Rate')
axes[1].set_xticklabels(['Female','Male'], rotation=0)

df[df['Survived']==1]['Age'].dropna().hist(ax=axes[2], alpha=0.6, label='Survived', bins=20, color='green')
df[df['Survived']==0]['Age'].dropna().hist(ax=axes[2], alpha=0.6, label='Did not survive', bins=20, color='red')
axes[2].set_title('Age Distribution by Survival')
axes[2].set_xlabel('Age')
axes[2].legend()

plt.tight_layout()
plt.show()



---
## Step 2 — Feature Engineering
New features: `Title` (from Name), `FamilySize` (SibSp + Parch + 1), `IsAlone` (FamilySize == 1)


In [ ]:

def engineer_features(df):
    df = df.copy()

    # Extract title from Name (e.g. "Braund, Mr. Owen" → "Mr")
    # Map rare/foreign titles to 'Rare' to reduce sparsity
    df['Title'] = df['Name'].str.extract(r',\s*([A-Za-z]+)\.', expand=False)
    df['Title'] = df['Title'].apply(lambda t: t if t in {'Mr','Miss','Mrs','Master'} else 'Rare')

    # FamilySize = SibSp + Parch + 1 (includes the passenger themselves)
    df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
    df['IsAlone']    = (df['FamilySize'] == 1).astype(int)

    # Fill missing values
    df['Age']      = df.groupby('Title')['Age'].transform(lambda x: x.fillna(x.median()))
    df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])
    df['Fare']     = df['Fare'].fillna(df['Fare'].median())

    return df


df = engineer_features(df)
print('Missing values after engineering:')
print(df[['Age','Embarked','Fare','Title','FamilySize','IsAlone']].isnull().sum())


In [ ]:

# Visualise survival rate by engineered features (Title and FamilySize)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df.groupby('Title')['Survived'].mean().sort_values().plot(
    kind='bar', ax=axes[0], color='steelblue', edgecolor='black')
axes[0].set_title('Survival Rate by Title')
axes[0].set_ylabel('Survival Rate')
axes[0].tick_params(axis='x', rotation=0)

df.groupby('FamilySize')['Survived'].mean().plot(
    kind='bar', ax=axes[1], color='coral', edgecolor='black')
axes[1].set_title('Survival Rate by Family Size')
axes[1].set_ylabel('Survival Rate')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()


---
## Step 3 — Encode Categorical Variables and Build Feature Matrix

In [ ]:

# ── Select features and one-hot encode categoricals ──────────────────────────
# Drop Cabin (high missingness), PassengerId/Name/Ticket (identifiers, no signal)
# drop_first=True removes one redundant dummy column per category
feature_cols = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare',
                'Embarked', 'Title', 'FamilySize', 'IsAlone']

df_encoded = pd.get_dummies(df[feature_cols].copy(),
                            columns=['Sex', 'Embarked', 'Title'],
                            drop_first=True)

print(f'Feature matrix shape: {df_encoded.shape}')
print(f'Columns: {list(df_encoded.columns)}')


In [ ]:

# ── Train / Validation / Test split (70 / 15 / 15) ───────────────────────────
X = df_encoded.values.astype(np.float32)
y = df['Survived'].values.astype(np.float32)

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)
X_val,   X_test, y_val,   y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)
print(f'Train: {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}')

# ── Z-score normalisation — stats computed from training set only ─────────────
X_mean = X_train.mean(axis=0)
X_std  = X_train.std(axis=0) + 1e-8
X_train = (X_train - X_mean) / X_std
X_val   = (X_val   - X_mean) / X_std
X_test  = (X_test  - X_mean) / X_std

# ── Convert to PyTorch tensors ────────────────────────────────────────────────
X_train_t = torch.tensor(X_train)
y_train_t = torch.tensor(y_train).view(-1, 1)
X_val_t   = torch.tensor(X_val)
y_val_t   = torch.tensor(y_val).view(-1, 1)
X_test_t  = torch.tensor(X_test)
y_test_t  = torch.tensor(y_test).view(-1, 1)

N_FEATURES = X_train_t.shape[1]
print(f'Number of input features: {N_FEATURES}')


---
## Step 4 — Reusable Training and Evaluation Helpers

In [ ]:

## Reusable training loop, evaluation, and plotting helpers
def train_model(model, optimizer, loss_fn, epochs=2000, print_every=200):
    train_losses, val_losses = [], []
    for epoch in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad()
        y_pred = model(X_train_t)
        loss   = loss_fn(y_pred, y_train_t)
        loss.backward()
        optimizer.step()
        train_losses.append(loss.item())

        model.eval()
        with torch.no_grad():
            val_pred = model(X_val_t)
            val_loss = loss_fn(val_pred, y_val_t)
            val_losses.append(val_loss.item())

        if epoch % print_every == 0:
            print(f'Epoch {epoch:>4}  Train Loss: {loss.item():.4f}  Val Loss: {val_loss.item():.4f}')

    return train_losses, val_losses


def evaluate_model(model, threshold=0.5):
    model.eval()
    with torch.no_grad():
        logits = model(X_test_t)
        probs  = torch.sigmoid(logits).numpy().flatten()
        preds  = (probs >= threshold).astype(int)
        true   = y_test_t.numpy().flatten().astype(int)
    acc  = accuracy_score(true, preds)
    prec = precision_score(true, preds, zero_division=0)
    rec  = recall_score(true, preds, zero_division=0)
    f1   = f1_score(true, preds, zero_division=0)
    cm   = confusion_matrix(true, preds)
    return {'Accuracy': acc, 'Precision': prec, 'Recall': rec, 'F1': f1,
            'confusion_matrix': cm, 'probs': probs, 'preds': preds, 'true': true}


def plot_losses(train_losses, val_losses, title):
    plt.figure(figsize=(8, 4))
    plt.plot(train_losses, label='Train loss', color='steelblue')
    plt.plot(val_losses,   label='Val loss',   color='coral')
    plt.xlabel('Epoch'); plt.ylabel('BCE Loss'); plt.title(title)
    plt.legend(); plt.grid(True); plt.tight_layout(); plt.show()


def plot_confusion(cm, title):
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Pred:0','Pred:1'], yticklabels=['True:0','True:1'])
    plt.title(title); plt.tight_layout(); plt.show()


---
## Task 1 — Linear Baseline (Logistic Regression)

A single `nn.Linear` layer with `BCEWithLogitsLoss` is mathematically equivalent to Logistic Regression. It learns one weight per feature and one bias — the simplest possible model.

In [ ]:

## Task 1 — Linear baseline model (Logistic Regression)
# nn.Linear(N_FEATURES, 1) + BCEWithLogitsLoss = Logistic Regression
# BCEWithLogitsLoss applies Sigmoid internally — more stable than manual Sigmoid + BCELoss
linear_model = nn.Linear(in_features=N_FEATURES, out_features=1)
loss_fn      = nn.BCEWithLogitsLoss()
optimizer    = torch.optim.Adam(linear_model.parameters(), lr=0.01)

print(linear_model)
print(f'Trainable parameters: {sum(p.numel() for p in linear_model.parameters())}')


In [ ]:
# Train the linear model
train_losses_lr, val_losses_lr = train_model(
    linear_model, optimizer, loss_fn, epochs=2000, print_every=400)
plot_losses(train_losses_lr, val_losses_lr, 'Task 1 — Logistic Regression: Loss Curves')

In [ ]:
# Evaluate the linear model on the held-out test set
results_lr = evaluate_model(linear_model)

print('── Logistic Regression Results ──────────────────')
for k, v in results_lr.items():
    if k not in ('confusion_matrix', 'probs', 'preds', 'true'):
        print(f'  {k:<12}: {v:.4f}')

plot_confusion(results_lr['confusion_matrix'], 'Task 1 — Logistic Regression Confusion Matrix')


---
## Task 2 — Non-linear MLP (Multi-Layer Perceptron)
Deeper network with ReLU activations and Dropout regularisation to learn non-linear boundaries.


In [ ]:

## Task 2 — Non-linear MLP with hidden layers and ReLU activations
# Dropout(0.3) randomly zeros 30 % of neurons during training to reduce overfitting
mlp_model = nn.Sequential(
    nn.Linear(N_FEATURES, 64), nn.ReLU(), nn.Dropout(0.3),
    nn.Linear(64, 32),         nn.ReLU(), nn.Dropout(0.3),
    nn.Linear(32, 16),         nn.ReLU(),
    nn.Linear(16, 1)
)
loss_fn_mlp   = nn.BCEWithLogitsLoss()
optimizer_mlp = torch.optim.Adam(mlp_model.parameters(), lr=0.001)

print(mlp_model)
print(f'Trainable parameters: {sum(p.numel() for p in mlp_model.parameters())}')


In [ ]:
# Train the MLP model
train_losses_mlp, val_losses_mlp = train_model(
    mlp_model, optimizer_mlp, loss_fn_mlp, epochs=2000, print_every=400)
plot_losses(train_losses_mlp, val_losses_mlp, 'Task 2 — MLP: Loss Curves')

In [ ]:
# Evaluate the MLP on the held-out test set
results_mlp = evaluate_model(mlp_model)

print('── MLP Results ──────────────────────────────────')
for k, v in results_mlp.items():
    if k not in ('confusion_matrix', 'probs', 'preds', 'true'):
        print(f'  {k:<12}: {v:.4f}')

plot_confusion(results_mlp['confusion_matrix'], 'Task 2 — MLP Confusion Matrix')


---
## Task 3 — Model Comparison
Metrics: Accuracy, Precision, Recall, F1-score


In [ ]:

## Task 3 — Side-by-side metrics comparison
metrics    = ['Accuracy', 'Precision', 'Recall', 'F1']
lr_scores  = [results_lr[m]  for m in metrics]
mlp_scores = [results_mlp[m] for m in metrics]

x = np.arange(len(metrics))
width = 0.35
fig, ax = plt.subplots(figsize=(9, 5))
bars1 = ax.bar(x - width/2, lr_scores,  width, label='Logistic Regression', color='steelblue', edgecolor='black')
bars2 = ax.bar(x + width/2, mlp_scores, width, label='MLP',                 color='coral',     edgecolor='black')

for bar in bars1 + bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)

ax.set_xticks(x); ax.set_xticklabels(metrics); ax.set_ylim(0, 1.05)
ax.set_ylabel('Score'); ax.set_title('Task 3 — Logistic Regression vs MLP')
ax.legend(); ax.grid(axis='y', alpha=0.4); plt.tight_layout(); plt.show()

# Print summary table
print(f'{"Metric":<12}  {"Logistic Reg":>14}  {"MLP":>10}')
print('-' * 42)
for m, lr, ml in zip(metrics, lr_scores, mlp_scores):
    better = '← MLP better' if ml > lr else ('← LR better' if lr > ml else '')
    print(f'{m:<12}  {lr:>14.4f}  {ml:>10.4f}  {better}')



---
## Task 4 — Feature Engineering Impact
Compare MLP trained on raw features vs engineered features (Title, FamilySize, IsAlone).


In [ ]:

## Task 4 — Raw features baseline (no Title, FamilySize, IsAlone)
# Same split and normalisation as the main pipeline
raw_cols = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']
df_raw   = pd.get_dummies(df[raw_cols].copy(), columns=['Sex','Embarked'], drop_first=True).astype('float32')

X_raw = df_raw.values.astype(np.float32)
y_raw = df['Survived'].values.astype(np.float32)

Xr_train, Xr_temp, yr_train, yr_temp = train_test_split(X_raw, y_raw, test_size=0.30, random_state=42, stratify=y_raw)
Xr_val,   Xr_test, yr_val,   yr_test = train_test_split(Xr_temp, yr_temp, test_size=0.50, random_state=42, stratify=yr_temp)

Xr_mean = Xr_train.mean(axis=0)
Xr_std  = Xr_train.std(axis=0) + 1e-8
Xr_train = (Xr_train - Xr_mean) / Xr_std
Xr_val   = (Xr_val   - Xr_mean) / Xr_std
Xr_test  = (Xr_test  - Xr_mean) / Xr_std

Xr_train_t = torch.tensor(Xr_train); yr_train_t = torch.tensor(yr_train).view(-1, 1)
Xr_val_t   = torch.tensor(Xr_val);   yr_val_t   = torch.tensor(yr_val).view(-1, 1)
Xr_test_t  = torch.tensor(Xr_test);  yr_test_t  = torch.tensor(yr_test).view(-1, 1)

N_RAW = Xr_train_t.shape[1]
print(f'Raw features: {list(df_raw.columns)}  ({N_RAW} total)')


In [ ]:

# Train an identical MLP on raw features; temporarily swap global tensors
def make_mlp(n_in):
    return nn.Sequential(
        nn.Linear(n_in, 64), nn.ReLU(), nn.Dropout(0.3),
        nn.Linear(64, 32),   nn.ReLU(), nn.Dropout(0.3),
        nn.Linear(32, 16),   nn.ReLU(),
        nn.Linear(16, 1)
    )

torch.manual_seed(42)
mlp_raw = make_mlp(N_RAW)
opt_raw = torch.optim.Adam(mlp_raw.parameters(), lr=0.001)
fn_raw  = nn.BCEWithLogitsLoss()

X_train_t_bak, y_train_t_bak = X_train_t, y_train_t
X_val_t_bak,   y_val_t_bak   = X_val_t,   y_val_t
X_test_t_bak,  y_test_t_bak  = X_test_t,  y_test_t

X_train_t, y_train_t = Xr_train_t, yr_train_t
X_val_t,   y_val_t   = Xr_val_t,   yr_val_t
X_test_t,  y_test_t  = Xr_test_t,  yr_test_t

print('Training MLP on RAW features only...')
train_model(mlp_raw, opt_raw, fn_raw, epochs=2000, print_every=500)
results_raw = evaluate_model(mlp_raw)

X_train_t, y_train_t = X_train_t_bak, y_train_t_bak
X_val_t,   y_val_t   = X_val_t_bak,   y_val_t_bak
X_test_t,  y_test_t  = X_test_t_bak,  y_test_t_bak

print('\n── Raw Features MLP ──')
for k, v in results_raw.items():
    if k not in ('confusion_matrix', 'probs', 'preds', 'true'):
        print(f'  {k:<12}: {v:.4f}')


In [ ]:

# Feature engineering impact — delta shows gain from engineered features
print(f'\n{"Metric":<12}  {"Raw features":>14}  {"+ Engineered":>14}  {"Delta":>8}')
print('-' * 55)
for m in metrics:
    raw_v = results_raw[m]
    eng_v = results_mlp[m]
    delta = eng_v - raw_v
    sign  = '+' if delta >= 0 else ''
    print(f'{m:<12}  {raw_v:>14.4f}  {eng_v:>14.4f}  {sign}{delta:>7.4f}')



---
## Summary
| Model | Notes |
|-------|-------|
| **Logistic Regression** | Linear boundary; fast; interpretable; good baseline |
| **MLP** | Non-linear; Dropout reduces overfitting; captures feature interactions |
| **Feature Engineering** | Title + FamilySize + IsAlone improve F1 and Recall |
